# SageMaker Coding Agent - Complete Version (AWS Bedrock)

A secure AI coding assistant powered by Amazon Bedrock Claude.

**Version: 2.0.0 (January 2025)**

## UI Layout (synced with compact versions)
```
Row 1: [Name] [Session▼] [📁Load] [+New] | [🌙Dark Mode]
Row 2: [📋Plan Mode] [☑Auto-Compact]
Chat:  HTML widget with internal scroll (fixes SageMaker drifting)
Row 3: [Send] [Clear] [Save] [Compact] [Status]
Row 4: Token usage display
```

## Features Implemented
- **UI**: HTML widget with internal scroll (fixes SageMaker drifting)
- **UI**: Auto-scroll to bottom (CSS flex-direction: column-reverse)
- **UI**: Dark mode toggle updates all existing messages
- **UI**: Session dropdown with Load/New buttons
- **Context**: Compact button (OpenCode-style 2-stage: prune + summarize)
- **Context**: Auto-Compact (ON by default, triggers at 90%)
- **Context**: Plan Mode (read-only exploration, creates implementation plan)
- **Session**: Save/Load sessions
- **Tools**: 15+ tools (file ops, bash, python, documents, semantic search)
- **Security**: Path validation, dangerous command filtering

## Not Implemented (vs OpenCode)
- Sub-agents
- MCP server integration
- Sliding window context

## Initialize Agent

In [ ]:
# === CONFIGURATION ===
# Edit these values to customize your agent

# Available Claude models in Bedrock (sorted by rate limit)
AVAILABLE_MODELS = {
    "Claude 3 Haiku (8 req/min)": "anthropic.claude-3-haiku-20240307-v1:0",
    "Claude 3 Sonnet (2 req/min)": "anthropic.claude-3-sonnet-20240229-v1:0",
    "Claude 3.5 Sonnet v2 (1 req/min)": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "Claude 3.5 Sonnet (1 req/min)": "anthropic.claude-3-5-sonnet-20240620-v1:0",
    "Claude 3 Opus": "anthropic.claude-3-opus-20240229-v1:0",
}

# Temperature presets
TEMPERATURE_OPTIONS = {
    "0.0 - Deterministic": 0.0,
    "0.3 - Low creativity": 0.3,
    "0.5 - Balanced": 0.5,
    "0.7 - Creative": 0.7,
    "1.0 - Maximum creativity": 1.0,
}

# Max output tokens
MAX_TOKENS_OPTIONS = {
    "2048 - Short": 2048,
    "4096 - Standard (Recommended)": 4096,
    "8192 - Long": 8192,
    "16384 - Very Long": 16384,
}

# Thinking budget (for extended thinking mode)
THINKING_BUDGET_OPTIONS = {
    "1024 - Quick": 1024,
    "2048 - Light": 2048,
    "4096 - Standard": 4096,
    "8192 - Deep": 8192,
    "16000 - Maximum": 16000,
}

# Region - Sydney (ap-southeast-2)
REGION = "ap-southeast-2"

# Workspace directory
WORKSPACE = "."

# === UI SETUP ===
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime

# Model selector
model_dropdown = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value="Claude 3 Haiku (8 req/min)",
    description='Model:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

# Temperature selector
temperature_dropdown = widgets.Dropdown(
    options=list(TEMPERATURE_OPTIONS.keys()),
    value="0.0 - Deterministic",
    description='Temperature:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

# Max tokens selector
max_tokens_dropdown = widgets.Dropdown(
    options=list(MAX_TOKENS_OPTIONS.keys()),
    value="4096 - Standard (Recommended)",
    description='Max Tokens:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

# Workspace input
workspace_input = widgets.Text(
    value=WORKSPACE,
    description='Workspace:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

# Extended thinking checkbox
thinking_checkbox = widgets.Checkbox(
    value=False,
    description='Enable Extended Thinking',
    indent=False,
    layout=widgets.Layout(width='200px')
)

# Thinking budget selector
thinking_budget_dropdown = widgets.Dropdown(
    options=list(THINKING_BUDGET_OPTIONS.keys()),
    value="4096 - Standard",
    description='Think Budget:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px'),
    disabled=True  # Disabled until thinking is enabled
)

# Mock mode checkbox
mock_checkbox = widgets.Checkbox(
    value=False,
    description='Mock Mode (no API calls)',
    indent=False
)

# Link thinking checkbox to budget dropdown
def on_thinking_change(change):
    thinking_budget_dropdown.disabled = not change['new']
    if change['new']:
        # Note: thinking mode requires temperature=1
        temperature_dropdown.value = "1.0 - Maximum creativity"
        temperature_dropdown.disabled = True
    else:
        temperature_dropdown.disabled = False

thinking_checkbox.observe(on_thinking_change, names='value')

# Layout
config_box = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 15px 0;">Agent Configuration</h3>'),
    model_dropdown,
    widgets.HTML(f'<div style="margin:5px 0 10px 105px;color:#666;font-size:12px;">Region: {REGION}</div>'),
    temperature_dropdown,
    max_tokens_dropdown,
    workspace_input,
    widgets.HTML('<hr style="margin:10px 0;">'),
    widgets.HBox([thinking_checkbox, thinking_budget_dropdown]),
    widgets.HTML('<div style="margin:5px 0 0 0;color:#888;font-size:11px;">Extended thinking uses more tokens but improves complex reasoning</div>'),
    widgets.HTML('<hr style="margin:10px 0;">'),
    mock_checkbox,
])

display(config_box)
print("\nConfigure settings and run the next cell to initialize.")

In [ ]:
# Initialize agent with selected configuration
from config import AgentConfig
from core.bedrock_client import BedrockClient
from core.tools import ToolRegistry, ToolContext
from core.agent_loop import AgentLoop
from core.security import SecurityManager, SecurityConfig
from core.audit import AuditLogger
from core.permissions import PermissionManager, PermissionResult
from core.memory import SessionManager
from core.context_manager import ContextManager
from core.project_config import ProjectConfig
from core.prompts import PromptBuilder

# Import tools
from tools import ALL_TOOLS

# Get selected settings from UI
selected_model_name = model_dropdown.value
selected_model_id = AVAILABLE_MODELS[selected_model_name]
selected_temperature = TEMPERATURE_OPTIONS[temperature_dropdown.value]
selected_max_tokens = MAX_TOKENS_OPTIONS[max_tokens_dropdown.value]
selected_workspace = workspace_input.value
thinking_enabled = thinking_checkbox.value
thinking_budget = THINKING_BUDGET_OPTIONS[thinking_budget_dropdown.value]
mock_mode = mock_checkbox.value

# Store settings globally for use in agent
SETTINGS = {
    "model_id": selected_model_id,
    "model_name": selected_model_name,
    "temperature": selected_temperature,
    "max_tokens": selected_max_tokens,
    "thinking_enabled": thinking_enabled,
    "thinking_budget": thinking_budget,
    "mock_mode": mock_mode,
}

print("=== Configuration ===")
print(f"Model: {selected_model_name}")
print(f"Model ID: {selected_model_id}")
print(f"Region: {REGION}")
print(f"Temperature: {selected_temperature}")
print(f"Max Tokens: {selected_max_tokens}")
print(f"Thinking Mode: {'Enabled' if thinking_enabled else 'Disabled'}")
if thinking_enabled:
    print(f"Thinking Budget: {thinking_budget} tokens")
print(f"Workspace: {selected_workspace}")
print(f"Mock Mode: {mock_mode}")

# Load base configuration and override with UI selections
config = AgentConfig.load()
config.region = REGION
config.primary_model = selected_model_id
config.workspace_root = selected_workspace
config.max_tokens = selected_max_tokens

# Initialize workspace path
workspace = os.path.abspath(config.workspace_root)

# Security
security_config = SecurityConfig(
    workspace_root=workspace,
    max_file_size=config.max_file_size,
    max_output_size=config.max_output_size,
    allow_network=config.allow_network,
)
security = SecurityManager(security_config)

# Audit logging
audit = AuditLogger(config.audit_dir)

# Session management
sessions = SessionManager(config.sessions_dir)

# Context management
context_mgr = ContextManager(workspace)

# Project config
project_config = ProjectConfig(workspace)

# Bedrock client (with mock mode support)
client = BedrockClient(config.primary_model, config.region, mock_mode=mock_mode)

# Tool registry
registry = ToolRegistry()
registry.register_all(ALL_TOOLS)

print(f"\n=== Initialized ===")
print(f"Workspace: {workspace}")
print(f"Tools registered: {len(registry.list_tools())}")
if project_config.has_instructions():
    print(f"Project instructions: {project_config.instruction_file}")

## Chat Interface

In [ ]:
# Global state
current_session = None
agent = None
pending_approval = None

# Plan Mode System Prompt (OpenCode-style)
PLAN_MODE_PROMPT = """You are in PLAN MODE. Your task is to EXPLORE and CREATE A PLAN, NOT execute.

# Plan Mode Rules
1. **READ-ONLY**: You can ONLY use these tools:
   - read_file, glob, grep, list_dir (explore codebase)
   - todo_write, todo_read (track what you're planning)

2. **NO WRITES**: Do NOT use:
   - write_file, edit_file, bash, python_exec, create_word, create_excel

3. **OUTPUT**: Create a detailed plan in your response:
   - What needs to be done (steps)
   - Which files need to be modified
   - What changes will be made
   - Any risks or considerations

4. **FORMAT**: End your response with a plan summary like:
   ```
   ## Implementation Plan
   1. [Step 1]
   2. [Step 2]
   ...

   ## Files to Modify
   - file1.py: [changes]
   - file2.py: [changes]

   ## Ready to Execute?
   Turn off Plan Mode and send "execute plan" to proceed.
   ```

Remember: EXPLORE and PLAN only. No modifications!
"""

# Simple Compactor class (inline for complete version)
class SimpleCompactor:
    PRUNE_PROTECT_TOKENS = 40000
    PRUNE_MIN_SAVINGS = 10000
    
    @classmethod
    def estimate_tokens(cls, text):
        return len(str(text)) // 4
    
    @classmethod
    def create_summary_prompt(cls):
        """Create summary prompt using Claude Code's 9-section format."""
        return """<analysis>
First, analyze the conversation to identify: main goal, technical concepts, files touched, errors encountered, and current progress.
</analysis>

Create a detailed summary following these EXACT sections:

1. **Primary Request and Intent**: What did the user explicitly ask for? What is their underlying goal?

2. **Key Technical Concepts**: Technologies, frameworks, libraries, patterns discussed or used.

3. **Files and Code Sections**: For each important file:
   - File path (absolute)
   - WHY it's important
   - What changes were made (if any)
   - Key code snippets (if relevant)

4. **Errors and Fixes**: For each error encountered:
   - The error message
   - How it was fixed
   - Any user feedback on the fix

5. **Problem Solving**: Problems solved during the session, and any ongoing issues still unresolved.

6. **ALL User Messages**: List EVERY user message verbatim (this prevents intent drift):
   - "message 1 exact text"
   - "message 2 exact text"
   - (continue for all messages)

7. **Pending Tasks**: Tasks mentioned but not yet completed.

8. **Current Work**: Precise current state including:
   - What step we're on
   - What was just completed
   - Relevant code context

9. **Next Step**: Only if directly in line with user's explicit request. Include direct quotes from user if applicable.

Format as a comprehensive summary that preserves all context needed to continue seamlessly."""
    
    @classmethod
    def prune_tool_outputs(cls, messages, max_tokens):
        """Prune old tool outputs, keep recent ones."""
        pruned = []
        total_protected = 0
        tokens_saved = 0
        
        for msg in reversed(messages):
            if msg.get("role") == "user" and isinstance(msg.get("content"), list):
                new_content = []
                for item in msg["content"]:
                    if isinstance(item, dict) and item.get("type") == "tool_result":
                        result = item.get("content", "")
                        tokens = cls.estimate_tokens(result)
                        if total_protected + tokens <= cls.PRUNE_PROTECT_TOKENS:
                            total_protected += tokens
                            new_content.append(item)
                        else:
                            tokens_saved += tokens
                            new_content.append({
                                "type": "tool_result",
                                "tool_use_id": item.get("tool_use_id"),
                                "content": f"[... {len(result)} chars pruned ...]"
                            })
                    else:
                        new_content.append(item)
                pruned.insert(0, {"role": "user", "content": new_content})
            else:
                pruned.insert(0, msg)
        
        return pruned, tokens_saved
    
    @classmethod
    def compact(cls, messages, summary):
        """Keep summary + last 5 messages."""
        summary_msg = {"role": "assistant", "content": f"[CONVERSATION SUMMARY]\n{summary}\n[END SUMMARY]"}
        last_messages = messages[-5:] if len(messages) > 5 else messages
        return [summary_msg] + last_messages

COMPACTOR = SimpleCompactor()

# UI State for message storage (SageMaker UI fix)
ui_state = {
    "messages": [],  # Store as tuples: (role, content, tool_name, timestamp)
    "dark_mode": True,  # Default to dark mode for SageMaker
}

# UI Components - HTML widget instead of Output (fixes SageMaker drifting)
chat_display = widgets.HTML(value='')
input_box = widgets.Textarea(
    placeholder='Type your message here...',
    layout=widgets.Layout(width='100%', height='80px')
)

# Session controls
session_name_input = widgets.Text(placeholder='Session name (optional)', layout=widgets.Layout(width='180px'))
session_dropdown = widgets.Dropdown(description='', options=[('New Session', None)], layout=widgets.Layout(width='200px'))
load_button = widgets.Button(description='Load', button_style='info', icon='folder-open')
new_button = widgets.Button(description='New', button_style='success', icon='plus')

# Action buttons
send_button = widgets.Button(description='Send', button_style='primary', icon='paper-plane')
clear_button = widgets.Button(description='Clear', button_style='warning', icon='trash')
save_button = widgets.Button(description='Save', button_style='info', icon='save')
compact_button = widgets.Button(description='Compact', button_style='', icon='compress', tooltip='Compress context')

# Status displays
status_label = widgets.HTML(value='<b style="color:#4caf50;">● Ready</b>')
tokens_label = widgets.HTML(value='<span style="color:#6a9955;font-size:12px;">Tokens: In:0 Out:0 | Total:0 | Calls:0</span>')

# Dark mode toggle
dark_toggle = widgets.ToggleButton(
    value=True,
    description='Dark Mode',
    icon='moon',
    button_style='',
    layout=widgets.Layout(width='120px')
)

# Plan Mode toggle (OpenCode-style)
plan_mode_toggle = widgets.ToggleButton(
    value=False,
    description='Plan Mode',
    icon='map',
    button_style='',
    tooltip='When ON: Agent only reads/explores, creates plan. When OFF: Normal execution.',
    layout=widgets.Layout(width='120px')
)

# Auto-compact checkbox (ON by default)
auto_compact_checkbox = widgets.Checkbox(
    value=True,
    description='Auto-Compact',
    indent=False,
    tooltip='Automatically compact when context exceeds 90%'
)

# Approval dialog (not used with auto-allow, but kept for compatibility)
approval_output = widgets.Output()
approve_button = widgets.Button(description='Approve', button_style='success')
deny_button = widgets.Button(description='Deny', button_style='danger')
approval_box = widgets.VBox([approval_output, widgets.HBox([approve_button, deny_button])])
approval_box.layout.display = 'none'

# Tool icons
TOOL_ICONS = {
    'read_file': '📖', 'write_file': '📝', 'edit_file': '✏️',
    'glob': '🔍', 'grep': '🔎', 'list_dir': '📁',
    'bash': '💻', 'python_exec': '🐍',
    'create_word': '📄', 'create_excel': '📊', 'create_markdown': '📋',
    'view_image': '🖼️', 'semantic_search': '🧠',
    'todo_write': '✅', 'todo_read': '📋',
}

def escape_html(text):
    return str(text).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

def render_chat():
    """Render all messages into HTML widget with internal scroll (SageMaker fix)."""
    dark = ui_state["dark_mode"]
    bg = '#1e1e1e' if dark else '#ffffff'
    fg = '#e0e0e0' if dark else '#333333'
    border = '#444' if dark else '#ccc'
    
    msgs_html = []
    for role, content, tool_name, ts in ui_state["messages"]:
        c = escape_html(content).replace('\\n', '<br>')
        
        if role == 'user':
            msgs_html.append(f'''<div style="padding:10px;margin:5px 0;border-left:3px solid #26c6da;">
                <b style="color:#26c6da;">[{ts}] You:</b>
                <div style="white-space:pre-wrap;margin:5px 0;color:{fg};">{c}</div>
            </div>''')
        elif role == 'assistant':
            msgs_html.append(f'''<div style="padding:10px;margin:5px 0;border-left:3px solid #42a5f5;">
                <b style="color:#42a5f5;">[{ts}] Agent:</b>
                <div style="white-space:pre-wrap;margin:5px 0;color:{fg};font-family:monospace;">{c}</div>
            </div>''')
        elif role == 'tool':
            icon = TOOL_ICONS.get(tool_name, '🔧')
            truncated = c[:2000] + '...' if len(content) > 2000 else c
            tool_bg = '#3d3222' if dark else '#fff8e1'
            tool_fg = '#f0d080' if dark else '#f57c00'
            msgs_html.append(f'''<details style="background:{tool_bg};padding:8px;margin:3px 0;border-radius:5px;font-size:12px;">
                <summary style="cursor:pointer;color:{tool_fg};"><b>{icon} {tool_name}</b></summary>
                <pre style="white-space:pre-wrap;font-size:11px;margin-top:5px;max-height:200px;overflow:auto;color:{fg};">{truncated}</pre>
            </details>''')
        elif role == 'system':
            sys_bg = '#4d2222' if dark else '#ffebee'
            sys_fg = '#ff8a80' if dark else '#c62828'
            msgs_html.append(f'''<div style="background:{sys_bg};color:{sys_fg};padding:8px;margin:3px 0;border-radius:5px;font-size:12px;">
                ⚠️ {c}
            </div>''')
    
    content = ''.join(msgs_html) if msgs_html else f'<p style="color:{fg};text-align:center;padding:20px;"><i>Type a message below to start chatting.</i></p>'
    
    # KEY: Scrollable div INSIDE the HTML content with flex-direction:column-reverse for auto-scroll
    chat_display.value = f'''<div style="
        height: 500px;
        max-height: 500px;
        overflow-y: auto;
        overflow-x: hidden;
        border: 1px solid {border};
        background: {bg};
        display: flex;
        flex-direction: column-reverse;
    "><div style="padding:10px;font-family:system-ui,-apple-system,sans-serif;">
        {content}
    </div></div>'''

def add_message(role, content, tool_name=None):
    """Add message and re-render."""
    ts = datetime.now().strftime('%H:%M:%S')
    ui_state["messages"].append((role, content, tool_name, ts))
    render_chat()

def update_tokens(usage):
    """Update token display."""
    tokens_label.value = f'<span style="color:#6a9955;font-size:12px;">Tokens: In:{usage["total_input"]:,} Out:{usage["total_output"]:,} | Total:{usage["total_input"]+usage["total_output"]:,} | Calls:{usage["api_calls"]}</span>'

def update_session_list():
    """Refresh session dropdown."""
    recent = sessions.list_sessions()[:15]
    options = [('New Session', None)]
    for s in recent:
        title = s['title'][:25] + ('...' if len(s['title']) > 25 else '')
        options.append((f"📁 {title}", s['id']))
    session_dropdown.options = options

def on_dark_toggle(change):
    """Handle dark mode toggle."""
    ui_state["dark_mode"] = change['new']
    render_chat()

dark_toggle.observe(on_dark_toggle, names='value')

def on_send(b):
    """Handle send button click."""
    global current_session, agent
    
    message = input_box.value.strip()
    if not message:
        return
    
    input_box.value = ''
    add_message('user', message)
    status_label.value = '<b style="color:#ff9800;">⋯ Processing...</b>'
    
    # Create session if needed
    if current_session is None:
        session_title = session_name_input.value.strip() if session_name_input.value.strip() else f"Chat: {message[:40]}"
        current_session = sessions.create(session_title)
        session_name_input.value = ''
    
    # Create tool context
    ctx = ToolContext(
        working_dir=workspace,
        session_id=current_session.id,
        security_manager=security,
        audit_logger=audit,
    )
    
    # Build system prompt (with Plan Mode if enabled)
    prompt_builder = PromptBuilder()
    system_prompt = prompt_builder.build(
        workspace_root=workspace,
        project_instructions=project_config.get_instructions(),
    )
    
    if plan_mode_toggle.value:
        add_message('system', '📋 PLAN MODE: Agent will explore and create a plan (no modifications)')
        system_prompt = system_prompt + "\\n\\n" + PLAN_MODE_PROMPT
    
    # Create agent
    agent = AgentLoop(
        client=client,
        registry=registry,
        system_prompt=system_prompt,
        context=ctx,
        max_turns=config.max_turns,
        doom_threshold=config.doom_loop_threshold,
        on_text=lambda t: add_message('assistant', t),
        on_tool_call=lambda n, i: None,  # Don't show calling status
        on_tool_result=lambda n, r: add_message('tool', r, n),
        on_tokens=update_tokens,
    )
    
    try:
        response = agent.run(message)
        
        # Save to session
        sessions.add_message(current_session, 'user', message)
        sessions.add_message(current_session, 'assistant', response)
        
        # Check context and auto-compact if needed
        status = context_mgr.get_status(agent.get_messages())
        pct = status['usage_percent'] * 100
        
        if auto_compact_checkbox.value and pct >= 90:
            add_message('system', f'🔄 Auto-compact triggered (context at {pct:.0f}%)...')
            try:
                messages = agent.get_messages()
                pruned_msgs, tokens_saved = COMPACTOR.prune_tool_outputs(messages, 200000)
                # Use 9-section summary prompt (available via COMPACTOR.create_summary_prompt())
                summary = "Conversation summary: " + "; ".join(
                    (m.get("content", "")[:60] if isinstance(m.get("content"), str) else "tool use")
                    for m in messages[:3]
                )
                compacted = COMPACTOR.compact(pruned_msgs, summary)
                # Note: AgentLoop may not support direct message replacement
                add_message('system', f'✅ Auto-compacted. Reduced from {len(messages)} to {len(compacted)} messages.')
            except Exception as e:
                add_message('system', f'Auto-compact failed: {e}')
        
        # Update status
        if pct >= 90:
            status_label.value = f'<b style="color:#f44336;">● Ready ({pct:.0f}% context - HIGH!)</b>'
        elif pct >= 75:
            status_label.value = f'<b style="color:#ff9800;">● Ready ({pct:.0f}% context)</b>'
        else:
            status_label.value = f'<b style="color:#4caf50;">● Ready ({pct:.0f}% context)</b>'
        
        if plan_mode_toggle.value:
            status_label.value = status_label.value.replace('Ready', '📋 Plan Mode')
        
    except Exception as e:
        add_message('system', f'Error: {str(e)}')
        status_label.value = '<b style="color:#f44336;">● Error</b>'

def on_clear(b):
    """Clear chat and start new session."""
    global current_session, agent
    current_session = None
    agent = None
    ui_state["messages"] = []
    tokens_label.value = '<span style="color:#6a9955;font-size:12px;">Tokens: In:0 Out:0 | Total:0 | Calls:0</span>'
    status_label.value = '<b style="color:#4caf50;">● Ready</b>'
    render_chat()

def on_save(b):
    """Save current session."""
    if current_session:
        sessions.save(current_session)
        add_message('system', f'Session saved: {current_session.id}')
        update_session_list()
    else:
        add_message('system', 'No session to save')

def on_load(b):
    """Load selected session."""
    global current_session, agent
    session_id = session_dropdown.value
    if not session_id:
        on_clear(None)
        return
    
    loaded = sessions.load(session_id)
    if loaded:
        current_session = loaded
        ui_state["messages"] = []
        add_message('system', f'Loaded session: {loaded.title}')
        update_session_list()
    else:
        add_message('system', f'Failed to load session: {session_id}')

def on_new(b):
    """Start new session."""
    global current_session, agent
    current_session = None
    agent = None
    ui_state["messages"] = []
    session_dropdown.value = None
    render_chat()
    status_label.value = '<b style="color:#4caf50;">● Ready (New)</b>'

def on_compact(b):
    """Manually compact context."""
    if not agent:
        add_message('system', 'No conversation to compact.')
        return
    
    add_message('system', 'Compacting conversation...')
    try:
        messages = agent.get_messages()
        pruned_msgs, tokens_saved = COMPACTOR.prune_tool_outputs(messages, 200000)
        if tokens_saved > 0:
            add_message('system', f'Stage 1: Pruned old tool outputs (~{tokens_saved:,} tokens saved)')
        
        # Use 9-section summary prompt (available via COMPACTOR.create_summary_prompt())
        summary = "Conversation summary: " + "; ".join(
            (m.get("content", "")[:60] if isinstance(m.get("content"), str) else "tool use")
            for m in messages[:3]
        )
        compacted = COMPACTOR.compact(pruned_msgs, summary)
        add_message('system', f'Compacted: {len(messages)} → {len(compacted)} messages. Click Save to persist.')
    except Exception as e:
        add_message('system', f'Compact failed: {e}')

send_button.on_click(on_send)
clear_button.on_click(on_clear)
save_button.on_click(on_save)
compact_button.on_click(on_compact)
load_button.on_click(on_load)
new_button.on_click(on_new)

# Layout - Row 1: [Name] [Session ▼] [📁Load] [+New] | [Model ▼]
header = widgets.HTML('<h2 style="color:#569cd6;margin:0 0 10px 0;">SageMaker Coding Agent v2.0.0</h2>')
row1 = widgets.HBox([
    session_name_input, session_dropdown, load_button, new_button,
    widgets.HTML('<span style="margin:0 10px;">|</span>'),
    dark_toggle
])
row2 = widgets.HBox([plan_mode_toggle, auto_compact_checkbox])
button_row = widgets.HBox([send_button, clear_button, save_button, compact_button, status_label])
tokens_row = widgets.HBox([tokens_label])
chat_box = widgets.VBox([header, row1, row2, chat_display, input_box, button_row, tokens_row])

# Initial render
update_session_list()
render_chat()

display(chat_box)

## Session Management

In [ ]:
# List recent sessions
recent = sessions.list_sessions()[:5]
if recent:
    print("Recent sessions:")
    for s in recent:
        print(f"  - {s['id']}: {s['title']} ({s['message_count']} messages)")
else:
    print("No previous sessions.")

In [ ]:
# Load a previous session (uncomment and set session_id)
# session_id = "20240101_120000"
# current_session = sessions.load(session_id)
# if current_session:
#     print(f"Loaded session: {current_session.title}")
#     for msg in current_session.messages[-5:]:
#         print(f"  [{msg['role']}]: {str(msg['content'])[:100]}...")

## Context Status

In [ ]:
# Check context and token status
if agent:
    # Context status
    status = context_mgr.get_status(agent.get_messages())
    print(f"Context usage: {status['usage_percent']:.1%}")
    print(f"Context tokens: {status['tokens']:,} / {status['max_tokens']:,}")
    print(f"Warning level: {status['warning_level']}")
    
    # Token usage
    tokens = agent.get_token_usage()
    print(f"\nAPI Token Usage:")
    print(f"  Input tokens: {tokens['input_tokens']:,}")
    print(f"  Output tokens: {tokens['output_tokens']:,}")
    print(f"  Total tokens: {tokens['total_tokens']:,}")
    print(f"  API calls: {tokens['api_calls']}")
else:
    print("No active conversation.")

## Audit Log

In [ ]:
# View audit log for current session
if current_session:
    entries = audit.get_session_log(current_session.id)
    print(f"Audit entries for session {current_session.id}: {len(entries)}")
    for e in entries[-5:]:
        print(f"  [{e['timestamp']}] {e['action']}: {e['tool_name']} - {e['result_summary'][:50]}")
else:
    print("No active session.")